In [1]:

# CELL 1 — FULL QAT TRAINING CODE


# NOTE: This cell defines:
# - EmotionPredictor
# - EmotionPredictorQuant
# - load_prepared_qat_for_inference()
# - evaluate()
# - predict_texts()
# - Dataset + label encoding functions
# - Everything required for QAT ***testing***
# -Training is commented out.

import os, shutil, time
from datetime import datetime
from typing import Tuple, List, Dict

import torch
import torch.nn as nn
import torch.ao.quantization as quant
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizerFast
from datasets import load_dataset
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# CONFIG
DATASET_NAME = "dair-ai/emotion"
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
DROPOUT = 0.3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



#  DATA + TOKENIZER UTILITIES


def setup_tokenizer():
    return BertTokenizerFast.from_pretrained(MODEL_NAME)

def load_and_prep_dataset(dataset_name=DATASET_NAME):
    ds = load_dataset(dataset_name)
    return ds["train"].to_pandas(), ds["validation"].to_pandas(), ds["test"].to_pandas()

def encode_labels(train_df, val_df, test_df):
    le = LabelEncoder()
    t = train_df.copy()
    v = val_df.copy()
    te = test_df.copy()

    t["label"]  = le.fit_transform(t["label"])
    v["label"]  = le.transform(v["label"])
    te["label"] = te["label"].map(lambda x: le.transform([x])[0])

    return t, v, te, le, len(le.classes_)


class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.max_len = max_len
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


def make_cpu_test_loader(test_df, tokenizer):
    ds = EmotionDataset(test_df, tokenizer, MAX_LEN)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)



# FP32 MODEL + QUANT MODEL DEFINITIONS


class EmotionPredictor(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled = out.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(pooled))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return loss, logits


class EmotionPredictorQuant(nn.Module):
    """
    Loads PREPARED-QAT model (fake quant).
    Does NOT call convert().
    Works on CPU or GPU.
    """
    def __init__(self, num_labels):
        super().__init__()
        self.num_labels = num_labels

        # Build backbone
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

        torch.backends.quantized.engine = "fbgemm"
        self.qconfig = quant.get_default_qat_qconfig("fbgemm")

        # do not quantize embedding
        self.bert.embeddings.qconfig = None

        quant.prepare_qat(self, inplace=True)

    def load_prepared_state(self, artifacts_dir, device="cpu"):
        sd = torch.load(os.path.join(artifacts_dir, "prepared_qat_state.pt"), map_location=device)
        self.load_state_dict(sd, strict=False)
        self.to(device)
        self.eval()
        print(f"[QAT] Loaded QAT model from {artifacts_dir} onto {device}")

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled = out.last_hidden_state[:, 0]
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return loss, logits



# QAT LOADER + METRICS + INFERENCE


def load_prepared_qat_for_inference(artifacts_dir, device="cpu"):
    meta = joblib.load(os.path.join(artifacts_dir, "meta.pkl"))
    tokenizer = BertTokenizerFast.from_pretrained(artifacts_dir)
    le = joblib.load(os.path.join(artifacts_dir, "label_encoder.pkl"))
    num_labels = meta["num_labels"]

    model = EmotionPredictorQuant(num_labels)
    model.load_prepared_state(artifacts_dir, device=device)
    return model, tokenizer, le


def evaluate(model, loader, device="cpu"):
    model.eval()
    tot_loss = 0
    all_preds, all_lbls = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, logits = model(batch["input_ids"], batch["attention_mask"], batch["labels"])
            tot_loss += loss.item()
            preds = logits.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_lbls.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(all_lbls, all_preds)
    f1 = f1_score(all_lbls, all_preds, average="macro")
    return tot_loss / len(loader), acc, f1


def predict_texts(model, tokenizer, label_encoder, texts, device="cpu"):
    enc = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        _, logits = model(enc["input_ids"], enc["attention_mask"])
    preds = logits.argmax(1).cpu().numpy()
    return label_encoder.inverse_transform(preds)



# QAT TRAINING + ARTIFACT SAVING


"""def run_qat_training_and_save(artifacts_dir="./qat_artifacts",
                              epochs_fp32=1,
                              epochs_qat=1):

    print("\n LOADING TOKENIZER + DATASET \n")
    tokenizer = setup_tokenizer()
    train_df, val_df, test_df = load_and_prep_dataset()
    train_df, val_df, test_df, label_encoder, num_labels = encode_labels(
        train_df, val_df, test_df
    )

    # Save label encoder + tokenizer
    os.makedirs(artifacts_dir, exist_ok=True)
    joblib.dump(label_encoder, os.path.join(artifacts_dir, "label_encoder.pkl"))
    tokenizer.save_pretrained(artifacts_dir)

    train_loader = DataLoader(
        EmotionDataset(train_df, tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        EmotionDataset(val_df, tokenizer, MAX_LEN),
        batch_size=BATCH_SIZE, shuffle=False
    )

    device = DEVICE


    # TRAIN FP32 BASE MODEL

    print("\n TRAINING FP32 MODEL \n")

    fp32_model = EmotionPredictor(num_labels).to(device)
    optimizer = torch.optim.AdamW(fp32_model.parameters(), lr=2e-5)

    for epoch in range(epochs_fp32):
        fp32_model.train()
        loop = tqdm(train_loader, desc=f"FP32 Epoch {epoch+1}")

        for batch in loop:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, _ = fp32_model(batch["input_ids"], batch["attention_mask"], batch["labels"])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loop.set_postfix(loss=loss.item())

        # Validation
        val_loss, val_acc, val_f1 = evaluate(fp32_model, val_loader, device)
        print(f"[FP32] Val Loss={val_loss:.4f} | Acc={val_acc:.4f} | F1={val_f1:.4f}")

    # Save FP32 checkpoint
    torch.save(fp32_model.state_dict(), os.path.join(artifacts_dir, "fp32_state.pt"))
    print("Saved FP32 checkpoint.")


    # PREPARE QAT MODEL

    print("\n PREPARING QAT MODEL \n")

    qat_model = EmotionPredictorQuant(num_labels).to(device)

    # Load FP32 into QAT model
    qat_model.load_state_dict(fp32_model.state_dict(), strict=False)

    optimizer = torch.optim.AdamW(qat_model.parameters(), lr=1e-5)


    # QAT TRAINING

    print("\nTRAINING QAT MODEL\n")

    for epoch in range(epochs_qat):
        qat_model.train()
        loop = tqdm(train_loader, desc=f"QAT Epoch {epoch+1}")

        for batch in loop:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, _ = qat_model(batch["input_ids"], batch["attention_mask"], batch["labels"])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loop.set_postfix(loss=loss.item())

        val_loss, val_acc, val_f1 = evaluate(qat_model, val_loader, device)
        print(f"[QAT] Val Loss={val_loss:.4f} | Acc={val_acc:.4f} | F1={val_f1:.4f}")


    #  SAVE QAT (PREPARED) STATE

    torch.save(
        qat_model.state_dict(),
        os.path.join(artifacts_dir, "prepared_qat_state.pt")
    )

    joblib.dump(
        {"num_labels": num_labels},
        os.path.join(artifacts_dir, "meta.pkl")
    )

    print("\n QAT ARTIFACTS SAVED TO:", artifacts_dir)

    # Optional: Create ZIP for Google Drive upload
    zip_path = artifacts_dir + ".zip"
    shutil.make_archive(artifacts_dir, 'zip', artifacts_dir)
    print(" Created ZIP file:", zip_path)

    return artifacts_dir, zip_path






run_qat_training_and_save(
    artifacts_dir="./qat_artifacts",
    epochs_fp32=1,
    epochs_qat=1
)

"""

'def run_qat_training_and_save(artifacts_dir="./qat_artifacts",\n                              epochs_fp32=1,\n                              epochs_qat=1):\n\n    print("\n LOADING TOKENIZER + DATASET \n")\n    tokenizer = setup_tokenizer()\n    train_df, val_df, test_df = load_and_prep_dataset()\n    train_df, val_df, test_df, label_encoder, num_labels = encode_labels(\n        train_df, val_df, test_df\n    )\n\n    # Save label encoder + tokenizer\n    os.makedirs(artifacts_dir, exist_ok=True)\n    joblib.dump(label_encoder, os.path.join(artifacts_dir, "label_encoder.pkl"))\n    tokenizer.save_pretrained(artifacts_dir)\n\n    train_loader = DataLoader(\n        EmotionDataset(train_df, tokenizer, MAX_LEN),\n        batch_size=BATCH_SIZE, shuffle=True\n    )\n    val_loader = DataLoader(\n        EmotionDataset(val_df, tokenizer, MAX_LEN),\n        batch_size=BATCH_SIZE, shuffle=False\n    )\n\n    device = DEVICE\n\n  \n    # TRAIN FP32 BASE MODEL\n   \n    print("\n TRAINING FP32 M

###INFERENCING

In [2]:

# CELL 2 — QAT TESTING PIPELINE


import gdown, zipfile, os, torch
from tqdm import tqdm

print("\n QAT MODEL TESTING PIPELINE \n")


#  DOWNLOAD QAT ARTIFACT ZIP FROM GOOGLE DRIVE

QAT_FILE_ID = "1b22RTDQD710XGAa3H40blUSvy4RUyqEN"
ZIP_FILE = "qat_artifacts.zip"

print(" Downloading QAT artifacts zip...")
url = f"https://drive.google.com/uc?id={QAT_FILE_ID}"
gdown.download(url, ZIP_FILE, quiet=False)
print(f" Downloaded ZIP: {ZIP_FILE}")


#  EXTRACT ARTIFACTS

ART_DIR = "./qat_artifacts"
os.makedirs(ART_DIR, exist_ok=True)

print("Extracting")
with zipfile.ZipFile(ZIP_FILE, "r") as z:
    for member in tqdm(z.infolist(), desc="Extracting", unit="files"):
        z.extract(member, ART_DIR)

print(f" Extracted artifacts to: {ART_DIR}")


# Load tokenizer + dataset

print(" Loading tokenizer + dataset")
tokenizer = setup_tokenizer()
train_df, val_df, test_df = load_and_prep_dataset()
train_df, val_df, test_df, label_encoder, num_labels = encode_labels(
    train_df, val_df, test_df
)
print(" Dataset ready.")

test_loader = make_cpu_test_loader(test_df, tokenizer)
print(" CPU test loader ready.")


#  LOAD QAT MODEL

print(" Loading QAT model")
device = torch.device("cpu")
model, tok, le = load_prepared_qat_for_inference(ART_DIR, device=device)

print("\n QAT Model loaded successfully\n")


 #Evaluate QAT Model

print(" QAT MODEL EVALUATION \n")

model.eval()
total_loss = 0
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating", unit="batch"):
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, logits = model(batch["input_ids"], batch["attention_mask"], batch["labels"])

        total_loss += loss.item()
        preds = logits.argmax(1).cpu().numpy()
        labels = batch["labels"].cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

# Compute metrics
from sklearn.metrics import accuracy_score, f1_score
test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average="macro")
test_loss = total_loss / len(test_loader)

print("\n QAT TEST METRICS")
print(f"QAT Test Loss:      {test_loss:.4f}")
print(f"QAT Test Accuracy:  {test_acc:.4f}")
print(f"QAT Test F1 Score:  {test_f1:.4f}")



#  Sample Predictions

print("\n SAMPLE PREDICTIONS (QAT) \n")

sample_texts = [
    "I feel so proud of myself!",
    "Everything feels terrible today.",
    "I’m confused and not sure how I feel.",
]

print("Running predictions")
preds = []
for txt in tqdm(sample_texts, desc="Predicting", unit="text"):
    p = predict_texts(model, tok, le, [txt], device="cpu")[0]
    preds.append(p)

# Print results
print("\nPredictions:")
for t, p in zip(sample_texts, preds):
    print(f"• \"{t}\" → {p}")

print("\n DONE \n")


 QAT MODEL TESTING PIPELINE 



Downloading...
From (original): https://drive.google.com/uc?id=1b22RTDQD710XGAa3H40blUSvy4RUyqEN
From (redirected): https://drive.google.com/uc?id=1b22RTDQD710XGAa3H40blUSvy4RUyqEN&confirm=t&uuid=84b90830-3975-4f2d-bfa6-9017835db0b1
To: /content/qat_artifacts.zip
100%|██████████| 814M/814M [00:08<00:00, 92.5MB/s]


 Downloaded ZIP: qat_artifacts.zip
📦 Extracting...


Extracting: 100%|██████████| 8/8 [00:12<00:00,  1.58s/files]


 Extracted artifacts to: ./qat_artifacts
 Loading tokenizer + dataset


...

[QAT] Loaded QAT model from ./qat_artifacts onto cpu

 QAT Model loaded successfully

 QAT MODEL EVALUATION 



Evaluating: 100%|██████████| 125/125 [17:25<00:00,  8.36s/batch]



 QAT TEST METRICS
QAT Test Loss:      0.7261
QAT Test Accuracy:  0.7545
QAT Test F1 Score:  0.5220

 SAMPLE PREDICTIONS (QAT) 

Running predictions


Predicting: 100%|██████████| 3/3 [00:04<00:00,  1.58s/text]


Predictions:
• "I feel so proud of myself!" → 1
• "Everything feels terrible today." → 0
• "I’m confused and not sure how I feel." → 4

 DONE 

